In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
# Load datasets 
cust = pd.read_csv('customer.csv') 
prod = pd.read_csv('products.csv') 
sales = pd.read_csv('sales.csv')

Phase 1: Data Cleaning

In [ ]:
# Explore 
print(cust.info()) 
print(cust.describe()) 
print(cust.isnull().sum())

In [ ]:
print(prod.info()) 
print(prod.describe()) 
print(prod.isnull().sum())

In [ ]:
print(sales.info()) 
print(sales.describe()) 
print(sales.isnull().sum())

In [ ]:
# Age column - fill missing with median
cust.fillna({'age': cust['age'].median()}, inplace=True)

# Price column - fill missing with mean
prod.fillna({'price': prod['price'].mean()}, inplace=True)

# Join date column - forward fill
cust['join_date'] = pd.to_datetime(cust['join_date'], errors='coerce')
cust.fillna({'join_date': cust['join_date'].ffill()}, inplace=True)


In [ ]:
# Remove duplicates 
cust.drop_duplicates(inplace=True) 
prod.drop_duplicates(inplace=True) 
sales.drop_duplicates(inplace=True)

In [ ]:
# Convert dates 
sales['order_date'] = pd.to_datetime(sales['order_date']) 
cust['join_date'] = pd.to_datetime(cust['join_date'])

Phase 2: Data Merging

In [ ]:
# Merge datasets following merge give error while doing operatio as it have 2 quantity column so we have to give suffixes to avoid confusion and then we can do operation on it

# sales_customers = pd.merge(sales, cust, on='customer_id', how='inner') 
# sales_products = pd.merge(sales, prod, on='product_id', how='inner') 
# retail_df = pd.merge(sales_customers, sales_products, on='order_id', how='inner', suffixes=('_cust','_prod'))

In [ ]:
# Merge sales with customers
sales_customers = pd.merge(sales, cust, on='customer_id', how='inner')

# Merge sales with products
sales_products = pd.merge(sales, prod, on='product_id', how='inner')

# Final merge with suffixes to avoid confusion
retail_df = pd.merge(
    sales_customers, 
    sales_products, 
    on=['order_id','customer_id','product_id','quantity','order_date'], 
    how='inner', 
    suffixes=('_cust','_prod')
)

# Now quantity is preserved cleanly
# retail_df['total_sales'] = retail_df['price'] * retail_df['quantity']
# print(retail_df.head(10))


In [ ]:
# Calculate total sales
retail_df['total_sales'] = retail_df['price'] * retail_df['quantity'] 
print(retail_df.head(10))

3. DATA Analysis operations are below :-

In [ ]:
# Top 5 customers by total purchase 
top_customers = retail_df.groupby('name')['total_sales'].sum().sort_values(ascending=False).head(5) 
print("Top 5 Customers:\n", top_customers)

In [ ]:
# Sales by City 
sales_city = retail_df.groupby('city')['total_sales'].sum() 
print("Sales by City:\n", sales_city)

In [ ]:
# Sales by Category 
sales_category = retail_df.groupby('category')['total_sales'].sum() 
print("Sales by Category:\n", sales_category)

In [ ]:
# Monthly Sales Trend 
retail_df['month'] = retail_df['order_date'].dt.to_period('M') 
monthly_sales = retail_df.groupby('month')['total_sales'].sum() 
print("Monthly Sales Trend:\n", monthly_sales)

In [ ]:
# Most Sold Product (by quantity) 
most_sold_product = retail_df.groupby('product_name')['quantity'].sum().sort_values(ascending=False).head(1) 
print("Most Sold Product:\n", most_sold_product)

In [ ]:

# Average Age of Customers per City 
avg_age_city = retail_df.groupby('city')['age'].mean() 
print("Average Age per City:\n", avg_age_city)

Phase 4: Visualization

In [ ]:
# Bar Plot - Sales by City 
plt.figure(figsize=(8,5)) 
sns.barplot(x=sales_city.index, y=sales_city.values) 
plt.title("Sales by City") 
plt.show()

In [ ]:
# Count Plot - Orders per Category 
plt.figure(figsize=(8,5)) 
sns.countplot(x='category', data=retail_df) 
plt.title("Orders per Category") 
plt.show()

In [ ]:
# Line Plot - Monthly Sales Trend 
monthly_sales.plot(kind='line', marker='o', figsize=(8,5)) 
plt.title("Monthly Sales Trend") 
plt.xlabel("Month") 
plt.ylabel("Total Sales") 
plt.show()

In [ ]:
# Pie Chart - Category Contribution % 
plt.figure(figsize=(6,6)) 
sales_category.plot(kind='pie', autopct='%1.1f%%') 
plt.title("Category Contribution %") 
plt.show()

In [ ]:
# Boxplot - Price Distribution by Category 
plt.figure(figsize=(8,5)) 
sns.boxplot(x='category', y='price', data=retail_df) 
plt.title("Price Distribution by Category") 
plt.show()

In [ ]:
# Heatmap - Correlation Matrix 
plt.figure(figsize=(8,6)) 
sns.heatmap(retail_df[['age','price','quantity','total_sales']].corr(), annot=True, cmap='coolwarm') 
plt.title("Correlation Matrix") 
plt.show()

Phase 5: Business Insights

In [ ]:
# Highest revenue city
highest_city = sales_city.idxmax()
print("Highest Revenue City:", highest_city)

In [ ]:
# Best performing category
best_category = sales_category.idxmax()
print("Best Performing Category:", best_category)

In [ ]:
# Most valuable customer
most_valuable_customer = top_customers.index[0]
print("Most Valuable Customer:", most_valuable_customer)

In [ ]:
# Are high-price products selling more?
avg_sales_high_price = retail_df[retail_df['price'] > retail_df['price'].mean()]['total_sales'].mean()
avg_sales_low_price = retail_df[retail_df['price'] <= retail_df['price'].mean()]['total_sales'].mean()
print("High-price products avg sales:", avg_sales_high_price)
print("Low-price products avg sales:", avg_sales_low_price)


In [ ]:
# Peak sales month
peak_month = monthly_sales.idxmax()
print("Peak Sales Month:", peak_month)
